# 03 - Insights and Comparison

This notebook presents the final narrative: what the data says, what it means, and what to do with it.

Main questions:
- How do inequality outcomes compare across Norway, USA, and the Philippines?
- What context from welfare-related indicators helps explain differences?
- What policy-relevant conclusions are justified by this dataset?

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = BASE_DIR / 'database' / 'database.db'

with sqlite3.connect(DB_PATH) as conn:
    norway = pd.read_sql_query('SELECT * FROM norway_inequality_indicators_clean ORDER BY year', conn)
    usa = pd.read_sql_query('SELECT * FROM usa_clean', conn)
    ph = pd.read_sql_query("SELECT * FROM philippines_clean WHERE region = 'Philippines' LIMIT 1", conn)
    norway_services = pd.read_sql_query("SELECT * FROM norway_public_services WHERE TRIM(Land) IN ('Norge', 'Norway') LIMIT 1", conn)

ph_row = ph.iloc[0]
norway.tail(3)

In [ ]:
def parse_number(value):
    if value is None:
        return None
    text = str(value).strip().replace(' ', '').replace('%', '').replace(',', '.')
    cleaned = []
    for ch in text:
        if ch.isdigit() or ch in {'.', '-'}:
            cleaned.append(ch)
        else:
            break
    if not cleaned:
        return None
    try:
        return float(''.join(cleaned))
    except ValueError:
        return None

def usa_point(metric):
    row = usa[(usa['income_type'] == 'MONEY INCOME') & (usa['measure'] == metric)]
    if row.empty:
        return None
    row = row.iloc[0]
    if pd.notna(row['year_2024_estimate']):
        return float(row['year_2024_estimate']), 2024
    if pd.notna(row['year_2023_estimate']):
        return float(row['year_2023_estimate']), 2023
    return None

usa_gini = usa_point('Gini index of income inequality')
usa_p90p10 = usa_point('90th/10th percentile income ratio')
usa_lowest_quintile = usa_point('Lowest quintile')

In [ ]:
def parse_number(value):
    if value is None:
        return None
    text = str(value).strip().replace(' ', '').replace(',', '.').replace('%', '')
    if text in {'', 'nan', 'None', '-', '...'}:
        return None
    try:
        return float(text)
    except ValueError:
        return None

norway_latest = norway.iloc[-1]
norway_gini = float(norway_latest['gini_all_population']) if pd.notna(norway_latest['gini_all_population']) else None
norway_p90p10 = float(norway_latest['p90_p10_all_population']) if pd.notna(norway_latest['p90_p10_all_population']) else None

ph_gini = float(ph_row['gini_2023']) if pd.notna(ph_row['gini_2023']) else None
if pd.notna(ph_row.get('gini_2009')) and pd.notna(ph_row.get('gini_2023')):
    ph_improvement = float(ph_row['gini_2009']) - float(ph_row['gini_2023'])
else:
    ph_improvement = None

norway_welfare_proxy = None
if not norway_services.empty:
    vals = [parse_number(v) for v in norway_services[['Barnehage', 'Utdanning', 'Pleie og omsorg', 'Helse']].iloc[0].tolist()]
    valid_vals = [float(v) for v in vals if v is not None]
    if len(valid_vals) == 4:
        norway_welfare_proxy = sum(valid_vals)

headline = pd.DataFrame([
    {'country': 'Norway', 'latest_gini': norway_gini, 'latest_year': int(norway_latest['year']), 'p90_p10': norway_p90p10},
    {'country': 'USA', 'latest_gini': usa_gini[0] if usa_gini else None, 'latest_year': usa_gini[1] if usa_gini else None, 'p90_p10': usa_p90p10[0] if usa_p90p10 else None},
    {'country': 'Philippines', 'latest_gini': ph_gini, 'latest_year': 2023, 'p90_p10': None},
])
headline

In [ ]:
gini_ranked = headline[['country', 'latest_gini']].dropna().sort_values('latest_gini', ascending=False)

ax = gini_ranked.plot(kind='bar', x='country', y='latest_gini', figsize=(7, 4), legend=False, title='Latest Gini ranking (higher = more inequality)')
ax.set_ylabel('Gini coefficient')
ax.set_xlabel('Country')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

gini_ranked

In [ ]:
norway_trend = norway[['year', 'gini_all_population']].rename(columns={'gini_all_population': 'gini'}).copy()
norway_trend['country'] = 'Norway'

usa_trend = []
row = usa[(usa['income_type'] == 'MONEY INCOME') & (usa['measure'] == 'Gini index of income inequality')]
if not row.empty:
    row = row.iloc[0]
    if pd.notna(row['year_2023_estimate']):
        usa_trend.append({'year': 2023, 'gini': float(row['year_2023_estimate']), 'country': 'USA'})
    if pd.notna(row['year_2024_estimate']):
        usa_trend.append({'year': 2024, 'gini': float(row['year_2024_estimate']), 'country': 'USA'})

ph_trend = pd.DataFrame([
    {'year': 2009, 'gini': ph_row['gini_2009'], 'country': 'Philippines'},
    {'year': 2012, 'gini': ph_row['gini_2012'], 'country': 'Philippines'},
    {'year': 2015, 'gini': ph_row['gini_2015'], 'country': 'Philippines'},
    {'year': 2018, 'gini': ph_row['gini_2018'], 'country': 'Philippines'},
    {'year': 2021, 'gini': ph_row['gini_2021'], 'country': 'Philippines'},
    {'year': 2023, 'gini': ph_row['gini_2023'], 'country': 'Philippines'},
]).dropna()

all_trends = pd.concat([norway_trend, pd.DataFrame(usa_trend), ph_trend], ignore_index=True)
pivot = all_trends.pivot_table(index='year', columns='country', values='gini', aggfunc='mean')

ax = pivot.plot(figsize=(10, 5), marker='o', title='Gini trends over time')
ax.set_ylabel('Gini coefficient')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
welfare_context = pd.DataFrame([
    {'country': 'Norway', 'inequality_gini': norway_gini, 'welfare_proxy': norway_welfare_proxy, 'proxy_name': 'Public-services total'},
    {'country': 'USA', 'inequality_gini': usa_gini[0] if usa_gini else None, 'welfare_proxy': usa_lowest_quintile[0] if usa_lowest_quintile else None, 'proxy_name': 'Lowest quintile income share'},
    {'country': 'Philippines', 'inequality_gini': ph_gini, 'welfare_proxy': ph_improvement, 'proxy_name': 'Gini improvement since 2009'},
]).dropna(subset=['inequality_gini', 'welfare_proxy'])

ax = welfare_context.plot.scatter(x='welfare_proxy', y='inequality_gini', figsize=(7, 5), title='Welfare context vs inequality')
for _, r in welfare_context.iterrows():
    ax.annotate(r['country'], (r['welfare_proxy'], r['inequality_gini']), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Welfare proxy (country-specific units)')
ax.set_ylabel('Gini coefficient')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

welfare_context

## Policy Interpretation

- **Norway**: Lower inequality and broader welfare/public-service structure are consistent with a redistributive model.
- **USA**: Higher inequality and lower bottom-quintile income share suggest weaker equalizing outcomes in market income distribution.
- **Philippines**: National Gini has improved since 2009, but levels remain above Norway and close to USA levels in available snapshots.

## Limitations

- Welfare context proxies are not measured in the same units across countries.
- Time coverage differs: Norway has long series; USA has recent estimates; Philippines has spaced-year observations.
- Cross-country interpretation should emphasize direction and relative position, not strict causal claims.

## Final Conclusion

Across the available indicators, inequality is lowest in Norway, higher in the USA, and improving but still elevated in the Philippines.
A policy mix centered on redistribution and public services appears aligned with lower inequality outcomes, while context differences require careful, transparent comparison.